# VUN sanity check 

Loads the cleaned **QM9** and **ZINC-250k** and runs the VUN sanity check.

## Setup

In [ ]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
print("cwd:", os.getcwd())

## 1 · Load cleaned datasets from the Hub

`use_cache=True` pulls the cleaned `(smiles, y)` tables (public, token-free) and skips the
download + sanitize pass entirely.

In [ ]:
from dataset.qm9 import load_qm9
from dataset.zinc import load_zinc

qm9 = load_qm9(use_cache=True)
zinc = load_zinc(use_cache=True)
print("QM9 :", qm9["ds"].num_rows, "| ZINC:", zinc["ds"].num_rows)

## 2 · Dataset-row VUN sanity

Round-trips the **train** molecules through the featurizer (`smiles -> (X, E) ->
tensor_to_mol -> largest_fragment -> SMILES`) and scores VUN on both datasets. Expect
**Valid ≈99%** (QM9, report-only round-trip) / **≈100%** (ZINC, filtered) and **Unique ≈100%**.
Novelty is **0 by construction** (train scored against itself). This is the dataset baseline
a trained model's samples are later compared against.

In [ ]:
from dataset.featurize import smiles_to_tensor
from dataset.metrics import vun_from_graphs

SANITY_N = 20000   # subsample for speed; set None for the full dataset

def dataset_row(name, d):
    smis = list(d["ds"]["smiles"])
    sample = smis[:SANITY_N] if SANITY_N else smis
    graphs = [smiles_to_tensor(s, atom_vocab=d["atom_vocab"]) for s in sample]
    m = vun_from_graphs(graphs, train_smiles=smis, atom_vocab=d["atom_vocab"])
    print(f"{name}: valid={m['validity']:.4f}  unique={m['uniqueness']:.4f}  "
          f"(novelty={m['novelty']:.4f}, self-check)  | n={m['n_generated']:,}")

dataset_row("QM9 ", qm9)
dataset_row("ZINC", zinc)